# Few-shot retinal severity classification (public data)

Estimate a 24-component whitened representation from unlabeled training features. Give the classifier 5, 20 or 40 labeled examples per class. Compare nearest-centroid scores with shrinkage linear discriminant analysis on held-out cases. The comparison below uses 20 examples per class. RetinaMNIST provides 1,080 training and 400 official test images. Classes 2–4 form the positive group and classes 0–1 the negative group. The outcome is diabetic retinopathy severity. The connection to glaucoma research is the few-shot learning question.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'few-shot-retinal-grading-real-data-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'joblib==1.6.0']],check=True)

In [ ]:
(root / 'prepare.py').write_text('import hashlib\nimport os\nfrom pathlib import Path\n\nDATA = Path(os.environ.get("PORTFOLIO_DATA", Path(__file__).resolve().parent / "data"))\nfrom urllib.request import urlretrieve\n\np = DATA / "medmnist/retinamnist.npz"\np.parent.mkdir(parents=True, exist_ok=True)\nif not p.exists():\n    urlretrieve(\n        "https://zenodo.org/records/10519652/files/retinamnist.npz?download=1", p\n    )\nassert hashlib.md5(p.read_bytes()).hexdigest() == "bd4c0672f1bba3e3a89f0e4e876791e4"\n',encoding='utf8')
subprocess.run([sys.executable,'prepare.py'],cwd=root,check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom sklearn.metrics import (\n    average_precision_score,\n    roc_auc_score,\n)\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", ROOT / "data"))\nSEEDS = [13, 37, 61]\n\n\ndef save(obj, name):\n    folder = ROOT / "checkpoints"\n    folder.mkdir(exist_ok=True)\n    joblib.dump(obj, folder / (name + ".joblib"), compress=3)\n\n\ndef scores(y, p):\n    return {\n        "auroc": float(roc_auc_score(y, p)),\n        "average_precision": float(average_precision_score(y, p)),\n    }\n\n\ndef describe(images):\n    from scipy.ndimage import gaussian_filter\n\n    a = images.astype("float32") / 255.0\n    blocks = a.reshape(len(a), 4, 7, 4, 7, 3).mean((2, 4)).reshape(len(a), -1)\n    texture = []\n    for s in [1.0, 2.0]:\n        residual = np.abs(a - gaussian_filter(a, (0, s, s, 0)))\n        texture.append(residual.mean((1, 2)))\n        texture.append(residual.std((1, 2)))\n    return np.column_stack([blocks, *texture])\n\n\ndef dataset(seed):\n    with np.load(DATA / "medmnist/retinamnist.npz") as raw:\n        return (\n            describe(raw["train_images"]),\n            raw["train_labels"].ravel(),\n            describe(raw["test_images"]),\n            raw["test_labels"].ravel(),\n        )\n\n\nfrom sklearn.decomposition import PCA\nfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis\n\n\ndef run(seed):\n    x, y, z, t = dataset(seed)\n    y = y >= 2\n    t = t >= 2\n    enc = make_pipeline(\n        StandardScaler(), PCA(n_components=24, whiten=True, random_state=seed)\n    )\n    a = enc.fit_transform(x)\n    b = enc.transform(z)\n    rng = np.random.default_rng(seed)\n    output = {}\n    for n in [5, 20, 40]:\n        ids = np.concatenate(\n            [\n                rng.choice(np.flatnonzero(y == k), n, replace=False)\n                for k in [False, True]\n            ]\n        )\n        centers = np.stack([a[ids[y[ids] == k]].mean(0) for k in [False, True]])\n        distance = ((b[:, None] - centers) ** 2).sum(2)\n        lda = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto").fit(\n            a[ids], y[ids]\n        )\n        output[f"prototype_{n}"] = scores(t, distance[:, 0] - distance[:, 1])\n        output[f"shrinkage_{n}"] = scores(t, lda.predict_proba(b)[:, 1])\n        save((enc, centers, lda), f"models-{n}-{seed}")\n    return output\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())